In [ ]:
import sys
sys.path.insert(0, '../')

import numpy as np
import matplotlib.pyplot as plt

import requests


In [2]:
url  = "https://raw.githubusercontent.com/mledoze/countries/master/countries.json"
data = requests.get(url).json()

# --- build entity lists ---
countries, regions, subregions, languages = [], [], [], []
triples = []

cca3_to_name = {e['cca3']: e.get('name', {}).get('common', '') for e in data}

for entry in data:
    country   = entry.get('name', {}).get('common', '')
    region    = entry.get('region', '')
    subregion = entry.get('subregion', '')
    langs     = list(entry.get('languages', {}).values())
    borders   = entry.get('borders', [])  # list of cca3 codes
    currencies = list(entry.get('currencies', {}).keys())
    timezones  = entry.get('timezones', [])

    if not (country and region):
        continue
    countries.append(country)
    regions.append(region)
    if subregion:
        subregions.append(subregion)
    languages.extend(langs)
    triples.append((country, 'region',    region))
    triples.append((country, 'subregion', subregion) if subregion else None)
    for lang in langs:
        triples.append((country, 'speaks', lang))
    for b in borders:
        triples.append((country, 'borders', cca3_to_name.get(b, '')))
    for c in currencies:
        triples.append((country, 'currency', c))
    for tz in timezones:
        triples.append((country, 'timezone', tz))

triples = [t for t in triples if t]

countries   = sorted(set(countries))
regions     = sorted(set(regions))
subregions  = sorted(set(subregions))
languages   = sorted(set(languages))
borders_list  = sorted(set(countries))   # country→country
currencies    = sorted(set(t[2] for t in triples if t[1] == 'currency'))
timezones     = sorted(set(t[2] for t in triples if t[1] == 'timezone'))

c_idx = {c: i for i, c in enumerate(countries)}
r_idx = {r: i for i, r in enumerate(regions)}
s_idx = {s: i for i, s in enumerate(subregions)}
l_idx = {l: i for i, l in enumerate(languages)}
cu_idx = {c: i for i, c in enumerate(currencies)}
tz_idx = {t: i for i, t in enumerate(timezones)}

NC, NR, NS, NL = len(countries), len(regions), len(subregions), len(languages)
print(f"Countries: {NC}, Regions: {NR}, Subregions: {NS}, Languages: {NL}")

# --- build relation matrices ---
R_loc    = np.zeros((NC, NR))   # country → region   (binary)
R_sub    = np.zeros((NC, NS))   # country → subregion
R_speaks = np.zeros((NC, NL))   # country → language
R_borders  = np.zeros((NC, NC))    # country × country
R_currency = np.zeros((NC, len(currencies)))
R_timezone = np.zeros((NC, len(timezones)))

for t in triples:
    country, rel, target = t
    if country not in c_idx:
        continue
    ci = c_idx[country]
    if rel == 'region'    and target in r_idx: R_loc[ci,    r_idx[target]] = 1.0
    if rel == 'subregion' and target in s_idx: R_sub[ci,    s_idx[target]] = 1.0
    if rel == 'speaks'    and target in l_idx: R_speaks[ci, l_idx[target]] = 1.0
    if rel == 'borders'  and target in c_idx:  R_borders[ci,  c_idx[target]]  = 1.0
    if rel == 'currency' and target in cu_idx: R_currency[ci, cu_idx[target]] = 1.0
    if rel == 'timezone' and target in tz_idx: R_timezone[ci, tz_idx[target]] = 1.0

print(f"R_loc    shape: {R_loc.shape}   density: {R_loc.mean():.3f}")
print(f"R_sub    shape: {R_sub.shape}  density: {R_sub.mean():.3f}")
print(f"R_speaks shape: {R_speaks.shape}  density: {R_speaks.mean():.3f}")

Countries: 250, Regions: 6, Subregions: 24, Languages: 155
R_loc    shape: (250, 6)   density: 0.167
R_sub    shape: (250, 24)  density: 0.041
R_speaks shape: (250, 155)  density: 0.011


In [3]:
from model import Lambert

relations = {
    'loc':    (R_loc,    regions),
    'sub':    (R_sub,    subregions),
    'speaks': (R_speaks, languages),
}

pipe = Lambert(
    entity_labels=countries,
    embed_temp=0.1,
    attn_temp=0.9,
    eps=1e-3
)
pipe.run(relations=relations, n_entities=len(countries))

print('emb_cat shape:', pipe.concept_space['emb'].shape)
print('categories:', len(pipe.explorer.categories))


getting embeddings...
embeddings ready: ['loc', 'sub', 'speaks']
exploring lattice...
Performing initial concept exploration . . .
  covered=0  categories=0
  covered=100  categories=15
Initial exploration phase complete.
Beginning full concept exploration . . .
Transitive closure reached.
exploration complete: emb shape=(250, 72), 72 categories
Identifying strongest defining traits . . . 
categories: 72
emb_cat shape: (250, 72)
categories: 72


In [5]:
# Forward
fwd = pipe.query(entities=["United Kingdom"])

# Backward  
bwd = pipe.query(features=["English"])




[forward]  5 results
  United Kingdom                  0.489
  Jersey                          0.489
  Guernsey                        0.489
  Isle of Man                     0.489
  Ireland                         0.489

  provenance (1 features):
    [speaks] English                          0.489

[backward]  10 results
  Zimbabwe                        0.489
  Zambia                          0.489
  Vanuatu                         0.489
  Tuvalu                          0.489
  Turks and Caicos Islands        0.489
  Tonga                           0.489
  United States                   0.489
  United Kingdom                  0.489
  United States Virgin Islands    0.489
  United States Minor Outlying Islands  0.489

  provenance (1 features):
    [speaks] English                          0.489
